# RONIN — Qwen3.5-4B training
Un único estudiante. LoRA BF16 con Unsloth. Selecciona GPU en Runtime → Change runtime type.
Descarga primero el artefacto ronin-qwen35-admitted-corpus-v1 desde GitHub Actions y súbelo cuando se solicite.

In [ ]:
!nvidia-smi
!pip -q install --upgrade unsloth unsloth_zoo datasets 'transformers>=5' trl

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU CUDA en Runtime → Change runtime type.')
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')
!git clone --depth 1 https://github.com/orbitalronin94/RONIN-IA.git /content/RONIN-IA

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Selecciona ronin-qwen35-admitted-corpus-v1.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content/ronin-corpus')
matches = list(Path('/content/ronin-corpus').rglob('train.txt'))
assert len(matches) == 1 and matches[0].stat().st_size > 0, matches
train_file = matches[0]
print('Corpus:', train_file, 'bytes:', train_file.stat().st_size)

In [ ]:
%cd /content/RONIN-IA
!python scripts/train_qwen35_unsloth.py --train-file {train_file} --steps 1000 --max-seq-length 2048 --eval-limit 140 --output-dir /content/ronin-qwen35-4b-output

In [ ]:
import json, shutil
from google.colab import files
summary = json.load(open('/content/ronin-qwen35-4b-output/summary.json', encoding='utf-8'))
print(json.dumps({k: summary[k] for k in ('model_id','steps','before_accuracy','after_accuracy','validation_improved')}, indent=2))
shutil.make_archive('/content/ronin-qwen35-4b-results', 'zip', '/content/ronin-qwen35-4b-output')
files.download('/content/ronin-qwen35-4b-results.zip')